# MTP-2 replication on Kaggle

1. Add this package as an input dataset (edit `PKG_IN` in cell 1 if its name differs).
2. Settings: Internet **on**, Accelerator **GPU T4 x2**.
3. Set `RUN_SAMPLE = True` (sample run, ~10–20 min) or `RUN_FULL = True` (full run, ~6–7 h), then run all cells.


In [ ]:
# 1. copy the package into the writable working folder
import os, shutil, glob
PKG_IN = "/kaggle/input/mtp-replication"      # <- change to the name of your attached dataset
src = glob.glob(os.path.join(PKG_IN, "**", "run_all.py"), recursive=True)
assert src, f"run_all.py not found under {PKG_IN}"
shutil.copytree(os.path.dirname(src[0]), "/kaggle/working/mtp-replication", dirs_exist_ok=True)
os.chdir("/kaggle/working/mtp-replication"); print(os.listdir("."))

In [ ]:
# 2. pinned packages (torch is preinstalled on Kaggle)
!pip install -q "transformers==5.16.1" "datasets==2.21.0" "pandas>=2.2,<3" accelerate sentencepiece protobuf pyreadr uharfbuzz

In [ ]:
# 3. Hugging Face token from Kaggle secrets (never written in the notebook)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF token: found")
except Exception as e:
    print("HF token: not set (gated tokenizers in the audit will be skipped)", e)

In [ ]:
MAKE_SUBMISSION = False   # author only: rebuilds the submission zip
FULL_DATA = False
if MAKE_SUBMISSION:
    !python scripts/export_xlsum.py --sample
    if FULL_DATA:
        !python scripts/export_xlsum.py
    shutil.make_archive("/kaggle/working/mtp-replication-submission", "zip", "/kaggle/working", "mtp-replication")
    print("submission zip: /kaggle/working/mtp-replication-submission.zip")

In [ ]:
RUN_SAMPLE = False
if RUN_SAMPLE:
    !python run_all.py --sample

In [ ]:
# 4. full reproduction (~6-7 h on GPU T4 x2). Set RUN_FULL = False if you only want the sample/submission zip.
RUN_FULL = False
if RUN_FULL:
    !python run_all.py --base /kaggle/working/mtp_final --zip

In [ ]:
if RUN_FULL:
    # 5. show the findings
    from IPython.display import Markdown, display
    display(Markdown(open("/kaggle/working/mtp_final/FINAL/FINAL_FINDINGS.md", encoding="utf-8").read().split("## Tables")[0]))
    display(Markdown(open("/kaggle/working/mtp_final/meco/MECO_FINDINGS.md", encoding="utf-8").read()))